# Laboratorio — Robot de entregas en un almacén (RESUELTO)

A partir de la imagen se construye el MDP y se resuelve con **Value Iteration** y **Policy Iteration**.

**Mapeo de la imagen al grid (5 filas × 6 columnas):**

| | c0 | c1 | c2 | c3 | c4 | c5 |
|---|---|---|---|---|---|---|
|**r0**|START|libre|libre|MURO|libre|ENTREGA +10 (T)|
|**r1**|libre|MURO|resbaloso|libre|peligro -3|libre|
|**r2**|libre|resbaloso|CARGA +2 (T)|libre|MURO|libre|
|**r3**|libre|libre|libre|resbaloso|libre|peligro -10 (T)|
|**r4**|libre|peligro -3|MURO|libre|libre|libre|


## Convención y notación

$$s=(row,col)\qquad T(s,a,s')=P(s'\mid s,a)\qquad R(s)$$

$$V_{k+1}(s)=R(s)+\gamma\max_a\sum_{s'}T(s,a,s')V_k(s')$$

$$V_{k+1}^{\pi}(s)=R(s)+\gamma\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')$$

### Acciones
```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```


## Parte 1 — Modela el MDP

Se completa la clase `WarehouseMDP` con las convenciones de la izquierda/derecha *relativas a la dirección de movimiento* (giro, no coordenadas absolutas del grid).

In [1]:
import numpy as np

UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)

class WarehouseMDP:
    def __init__(self, living_reward=-1.0, gamma=0.9, slip_intended=0.60):
        self.height = 5
        self.width = 6

        self.start = (0, 0)

        # Estanterias / paredes (a partir de la imagen)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Piso resbaloso (celdas amarillas)
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # Estados terminales: {(row, col): reward}
        self.terminal_states = {
            (0, 5): 10.0,   # ENTREGA
            (2, 2): 2.0,    # CARGA (estacion de carga)
            (3, 5): -10.0,  # PELIGRO MORTAL
        }

        # Peligros NO terminales
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = living_reward
        self.gamma = gamma
        self.slip_intended = slip_intended  # prob. de moverse en la direccion deseada en piso resbaloso

        self.actions = [UP, DOWN, LEFT, RIGHT]

        # Desviaciones (izquierda, derecha) RELATIVAS a la direccion en la que se mueve el robot
        self.perp = {
            UP:    (LEFT, RIGHT),
            DOWN:  (RIGHT, LEFT),
            LEFT:  (DOWN, UP),
            RIGHT: (UP, DOWN),
        }

    def is_valid_state(self, state):
        r, c = state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if (r, c) not in self.walls]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve [(next_state, probability), ...]
        - Un estado terminal es absorbente: se queda en si mismo con prob 1.
        - Las probabilidades dependen de si `state` es resbaloso.
        - Si el movimiento sale del grid o golpea una pared, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_intended = self.slip_intended
            p_dev = (1.0 - p_intended) / 2.0
        else:
            p_intended, p_dev = 0.90, 0.05

        left_dev, right_dev = self.perp[action]
        outcomes = [(action, p_intended), (left_dev, p_dev), (right_dev, p_dev)]

        result = {}
        for act, prob in outcomes:
            ns = (state[0] + act[0], state[1] + act[1])
            if not self.is_valid_state(ns):
                ns = state
            result[ns] = result.get(ns, 0.0) + prob

        return list(result.items())


### Validación mínima del modelo

Antes de implementar Bellman, se valida primero el MDP.

In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Numero de estados:", len(S))

for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("Todas las distribuciones de transicion suman 1.")


Numero de estados: 26
Todas las distribuciones de transicion suman 1.


## Parte 2 — Value Iteration

$$V_{k+1}(s)=R(s)+\gamma\max_a\sum_{s'}T(s,a,s')V_k(s')$$


In [3]:
def expected_next_value(grid, state, action, V):
    return sum(p * V[ns] for ns, p in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for i in range(max_iter):
        delta = 0.0
        V_new = {}
        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                q_values = [expected_next_value(grid, s, a, V) for a in grid.actions]
                V_new[s] = grid.get_reward(s) + grid.gamma * max(q_values)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            return V, i + 1
    return V, max_iter


def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        q_values = {a: expected_next_value(grid, s, a, V) for a in grid.actions}
        policy[s] = max(q_values, key=q_values.get)
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation
$$V_{k+1}^{\pi}(s)=R(s)+\gamma\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')$$

### Policy Improvement
$$\pi_{\mathrm{new}}(s)=\arg\max_a\sum_{s'}T(s,a,s')V^\pi(s')$$


In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}
    for i in range(max_iter):
        delta = 0.0
        V_new = {}
        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                a = policy[s]
                V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            break
    return V


def policy_improvement(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        q_values = {a: expected_next_value(grid, s, a, V) for a in grid.actions}
        policy[s] = max(q_values, key=q_values.get)
    return policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. politica inicial arbitraria (siempre UP)
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []
    for i in range(max_iter):
        # 2. evaluacion
        V = policy_evaluation(grid, policy, threshold=threshold)
        # 3. mejora
        new_policy = policy_improvement(grid, V)
        changed = sum(1 for s in new_policy if new_policy[s] != policy[s])
        history.append(changed)
        policy = new_policy
        # 4. repetir hasta estabilidad
        if changed == 0:
            break
    return policy, V, history


## Parte 4 — Visualización y comparación

In [5]:
ARROWS = {
    (-1, 0): "\u2191",
    ( 1, 0): "\u2193",
    ( 0,-1): "\u2190",
    ( 0, 1): "\u2192",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolitica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolitica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\nAmbos algoritmos encontraron la misma politica optima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Politica:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [16, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Politica:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  

## Parte 5 — Interpreta la política

**1. Desde `START`, ¿el robot busca la entrega +10 o prefiere la carga +2?**

Prefiere la **estación de carga (+2)**. Desde `(0,0)` la política óptima lo lleva hacia la derecha y luego hacia abajo, entrando por la celda resbalosa `(1,2)` hasta `(2,2)` (CARGA), en vez de continuar hacia `(0,5)` (ENTREGA).

**2. ¿Por qué una recompensa menor podría ser óptima?**

Porque el valor de un estado no depende solo de la recompensa terminal, sino de la recompensa terminal **descontada por la distancia** y **penalizada por los costos de paso y riesgos** en el camino. El único camino hacia ENTREGA obliga a pasar por la celda de peligro `(1,4)` con `R=-3` (los muros en `(0,3)` y `(2,4)` bloquean cualquier otra ruta), mientras que el camino a CARGA es más corto y no atraviesa peligros. Aun con descuento γ=0.9, el costo extra (pasos + el -3) hace que `+2` cercano y seguro vencer a `+10` lejano y riesgoso.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**

En estados adyacentes a una celda resbalosa donde, de no ser por el riesgo de "deslizarse" hacia una pared o hacia atrás, el camino más corto pasaría por ella igualmente sin cambio de rumbo; aquí el robot igual atraviesa `(1,2)`, `(2,1)` y `(3,3)` porque son la ruta más corta a CARGA (o para escapar del peligro), pero su *valor* baja frente al de un piso normal equivalente (compárese `V(1,2)` con estados no resbalosos a la misma distancia de una meta) — el robot no evita la celda porque no hay ruta alternativa más barata, pero sí "sabe" que vale menos.

**4. ¿Qué papel cumple el costo por paso -1?**

Es lo que introduce urgencia: penaliza cada paso adicional, empujando al robot hacia la meta *más cercana* con buen retorno, no necesariamente la de mayor recompensa nominal. Sin él (o con un costo muy pequeño), el robot estaría dispuesto a recorrer caminos más largos y arriesgados para llegar a +10.

**5. ¿Por qué T(s,a,s') ya no puede implementarse con las mismas probabilidades para todos los estados?**

Porque la dinámica depende del **tipo de piso** del estado actual: piso normal usa (0.90, 0.05, 0.05) y piso resbaloso usa (0.60, 0.20, 0.20). Por lo tanto `T` es una función que consulta `state in self.slippery_states` antes de fijar las probabilidades — ya no es una tabla fija (misma distribución) para cualquier estado, sino condicional al estado de origen.


### Experimento A — Menos costo por paso (`living_reward = -0.1`)

**Predicción antes de ejecutar:** con un costo por paso mucho menor, la urgencia baja: el robot podría preferir todavía CARGA (porque el camino a ENTREGA sigue cruzando el peligro -3), pero el valor de todos los estados debería subir considerablemente y el camino podría "relajarse" (aceptar rutas más largas si mejoran ligeramente el retorno).

**Resultado real:** el robot **sigue prefiriendo CARGA** desde `START`. El motivo es estructural, no de costo de paso: cruzar por `(1,4)` para llegar a ENTREGA cuesta -3 fijos que no dependen de `living_reward`, así que bajar el costo por paso no cambia la ventaja de CARGA (ver Bonus).


In [7]:
grid_A = WarehouseMDP(living_reward=-0.1)
V_A, n_A = value_iteration(grid_A)
pi_A = extract_policy(grid_A, V_A)
print("Iteraciones:", n_A)
print_values(grid_A, V_A)
print()
print_policy(grid_A, pi_A)
print("\nV(start) =", V_A[grid_A.start])


Iteraciones: 26
 +1.649 |  +1.992 |  +2.361 |   WALL   |  +8.591 | +10.000
 +1.358 |   WALL   |  +2.797 |  +3.911 |  +4.550 |  +8.591
 +1.259 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874

 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

V(start) = 1.6487099915120942


### Experimento B — Piso muy resbaloso (`0.60 → 0.40`)

**Predicción:** más incertidumbre en las celdas resbalosas debería bajar el valor de las rutas que pasan por ellas (`(1,2)`, `(2,1)`, `(3,3)`), quizás haciendo más atractivo un rodeo si existiera uno más seguro; pero como no hay ruta alternativa a CARGA que evite `(1,2)`/`(2,1)`, la política probablemente no cambia, solo bajan los valores.

**Resultado real:** la política es la misma; `V(start)` baja levemente (mayor riesgo de deslizarse hacia una pared o retroceder alarga el viaje esperado).


In [8]:
grid_B = WarehouseMDP(slip_intended=0.40)
V_B, n_B = value_iteration(grid_B)
pi_B = extract_policy(grid_B, V_B)
print("Iteraciones:", n_B)
print_values(grid_B, V_B)
print()
print_policy(grid_B, pi_B)
print("\nV(start) =", V_B[grid_B.start])


Iteraciones: 22
 -2.706 |  -1.809 |  -0.797 |   WALL   |  +7.607 | +10.000
 -2.652 |   WALL   |  +0.395 |  +2.104 |  +3.670 |  +7.607
 -1.744 |  -0.670 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.831 |  -0.774 |  +0.534 |  -1.137 |  -2.152 | -10.000
 -2.785 |  -3.929 |   WALL   |  -2.152 |  -2.974 |  -4.041

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

V(start) = -2.706200866141053


### Experimento C — Más paciencia (`gamma = 0.99`)

**Predicción:** con γ más cercano a 1, el robot descuenta menos las recompensas lejanas, así que ENTREGA (+10, lejos) se vuelve relativamente más atractiva frente a CARGA (+2, cerca). Podría cambiar la política en celdas intermedias.

**Resultado real:** el camino óptimo **sigue yendo a CARGA** desde `START`, pero la política en la celda `(1,2)` cambia de bajar hacia CARGA a moverse lateralmente explorando la ruta hacia ENTREGA — es decir, la mayor paciencia sí empieza a inclinar la balanza en algunas celdas, aunque no en `START`. El costo fijo del peligro -3 en el único camino a ENTREGA sigue pesando.


In [9]:
grid_C = WarehouseMDP(gamma=0.99)
V_C, n_C = value_iteration(grid_C)
pi_C = extract_policy(grid_C, V_C)
print("Iteraciones:", n_C)
print_values(grid_C, V_C)
print()
print_policy(grid_C, pi_C)
print("\nV(start) =", V_C[grid_C.start])


Iteraciones: 24
 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

V(start) = -1.456064938873927


### Bonus — ¿Existe un `living_reward` donde `START` cambia de CARGA a ENTREGA?

Se barre `living_reward` desde valores muy negativos (paso muy caro) hasta valores positivos (paso "gratis" o incluso recompensado) y se observa la política en `(0,2)`, el punto de la primera bifurcación real hacia CARGA vs. ENTREGA.


In [10]:
prev = None
for lr in np.arange(-3.0, 3.51, 0.1):
    g = WarehouseMDP(living_reward=lr)
    V, n = value_iteration(g)
    pi = extract_policy(g, V)
    a = ARROWS[pi[(0, 2)]]
    if a != prev:
        print(f"living_reward={lr:+.2f}  politica en (0,2) = {a}   V(0,2) = {V[(0,2)]:.3f}")
        prev = a


living_reward=-3.00  politica en (0,2) = ↓   V(0,2) = -5.858


living_reward=+2.00  politica en (0,2) = ↑   V(0,2) = 19.999


living_reward=+3.20  politica en (0,2) = ←   V(0,2) = 31.999
living_reward=+3.30  politica en (0,2) = ↑   V(0,2) = 32.999


**Conclusión del bonus:** para *cualquier* `living_reward ≤ 0` (cualquier costo de paso realista, incluso casi cero) la política **nunca cambia**: el robot siempre prefiere CARGA. Esto no es un problema de calibración del costo de paso, sino estructural: los muros en `(0,3)` y `(2,4)` obligan a que la **única** ruta hacia ENTREGA cruce la celda de peligro `(1,4)` con `R=-3`. Ese costo fijo de -3 (independiente de `living_reward`) hace que ENTREGA nunca compense frente al camino corto y seguro hacia CARGA, sin importar qué tan barato sea moverse.

Solo aparece un cambio de política en `(0,2)` cuando `living_reward` se vuelve positivo y grande (≈ +2 o más), es decir, cuando **moverse premia** en vez de costar — un escenario degenerado (el agente preferiría deambular indefinidamente) y no una respuesta realista a la pregunta planteada. Por eso la respuesta honesta es: **no existe un umbral realista de `living_reward` que haga óptimo ir a ENTREGA desde `START`**; lo que habría que cambiar es el layout (quitar el peligro del único camino) o la magnitud de la penalización de esa celda de peligro.
